# Statistic Multiple Dims NetCDF/Zarr to Region levels by shapefile Index

_This example was created by Zhaoquan YU (15/06/2025 at ClimSystems)._

## 1. Stats Global countries level

#### Setting

In [10]:
import numpy as np, xarray as xr, os, pandas as pd, geopandas as gpd
import os, requests; exec(requests.get(f'http://45.120.118.102:6888/lib/Jpkg?key=J.').json()['Jpkg'])

crs = "epsg:4326"
shp = gpd.read_file(r'M:\Project_Output_2025_10\MAM-Demo\Dummy_Polygons\info\info_shp\info_shp.shp').to_crs(crs) ############################ Modify as required
shp = shp.sort_values('source') ############################################################################################################# Modify as required
# shp = shp.sort_values('FID')
shp['SN'] = range(1, len(shp)+1)
# shp['geometry'] = shp['geometry'].buffer(0)

keep_cols = ['SN', 'POA_CODE21', 'source'] ########################################################## Modify as required

for col in keep_cols:
    shp[col] = shp[col].astype(str)

print('Invalid records:', shp.loc[~shp.geometry.is_valid])

##################################################################################################### Modify as required. If they are incorrect, see the details about zarr file structure below.
pths  = [5, 50, 95]
ssps  = [119, 126, 245, 370, 585]
years = [2005, 2030, 2040, 2050]

Invalid records: Empty GeoDataFrame
Columns: [POA_CODE21, POA_NAME21, AUS_CODE21, AUS_NAME21, AREASQKM21, LOCI_URI21, SHAPE_Leng, SHAPE_Area, SN, source, geometry]
Index: []


#### 1.1 Stats sub level for Data

In [11]:
# running (base) johnny@johnny-worker-01

vns = [
    "Extreme_Wind_Speed", ########################################################################### Modify as required
]

for vn in vns:
    input_path = f"http://192.168.1.218:9000/climatehub.climate.indicator.v1/Australia"  ##################### Modify as required
    output_path = r"Z:\works\Lucas\Data_Works"  ################################################################################# Modify as required
    os.makedirs(output_path, exist_ok=True)
    da = (
        xr.open_zarr(f"{input_path}/Australia_{vn}.zarr") ####################################################################### Modify as required
        .rename({"lon": "x", "lat": "y"})
        .rio.write_crs(crs)[vn]
    )

    print("Zarr 数据的结构和维度信息如下：")
    print(da)

    df_list = []
    ofn = f"{output_path}/{vn}_stats_sub_level.csv"

# for vn in vns:
#     # 1. 核心修改：如果是这个特定的变量，我们手动指定它在服务器上的真实 Zarr 文件夹名称
#     if vn == 'Water_Stress_Category':
#         zarr_name = 'Global_Water_Stress_Category_Aqueduct40.zarr'
#     else:
#         zarr_name = f'Global_{vn}.zarr'

#     input_path = f'http://192.168.1.218:9000/climatehub.climate.indicator.v1/Global'
#     output_path = r'Z:\works\Lucas\Data_Works'
#     os.makedirs(output_path, exist_ok=True)

#     # 2. 核心修改：使用刚刚判断好的 zarr_name 来打开数据，最后用 [vn] 提取变量
#     da = xr.open_zarr(f'{input_path}/{zarr_name}').rename({'lon':'x', 'lat':'y'}).rio.write_crs(crs)[vn]

#     print("==Zarr file's structure==")
#     print(da)

#     df_list = []
#     ofn = f'{output_path}/{vn}_stats_sub_level.csv'


if "cat" in da.dims:
    for ssp in ssps:
        for year in years:
            for pth in pths:
                for cat in da.cat.values.tolist():
                    print(vn, ssp, year, pth, cat)
                    df = edge_track_zonal_stats(
                        da.sel(ssp=ssp, year=year, pth=pth, cat=cat).squeeze(),
                        shp,
                        keep_cols=keep_cols,
                    )
                    df.insert(len(keep_cols), "Indicator", vn)
                    df.insert(len(keep_cols), "cat", cat)
                    df.insert(len(keep_cols), "pth", pth)
                    df.insert(len(keep_cols), "year", year)
                    df.insert(len(keep_cols), "ssp", ssp)
                    df_list.append(df)
else:
    for ssp in ssps:
        for year in years:
            for pth in pths:
                print(vn, ssp, year, pth)
                df = edge_track_zonal_stats(
                    da.sel(ssp=ssp, year=year, pth=pth).squeeze(),
                    shp,
                    keep_cols=keep_cols,
                )
                df.insert(len(keep_cols), "Indicator", vn)
                df.insert(len(keep_cols), "pth", pth)
                df.insert(len(keep_cols), "year", year)
                df.insert(len(keep_cols), "ssp", ssp)
                df_list.append(df)
pd.concat(df_list).to_csv(ofn, index=False)

Zarr 数据的结构和维度信息如下：
<xarray.DataArray 'Extreme_Wind_Speed' (ari: 1, pth: 3, ssp: 5, year: 20,
                                        y: 691, x: 886)> Size: 735MB
dask.array<open_dataset-Extreme_Wind_Speed, shape=(1, 3, 5, 20, 691, 886), dtype=float32, chunksize=(1, 3, 5, 20, 120, 30), chunktype=numpy.ndarray>
Coordinates:
  * ari          (ari) int64 8B 100
  * y            (y) float32 3kB -10.0 -10.05 -10.1 ... -44.4 -44.45 -44.5
  * x            (x) float32 4kB 112.0 112.0 112.1 112.1 ... 156.1 156.2 156.2
  * pth          (pth) int64 24B 5 50 95
  * ssp          (ssp) int64 40B 119 126 245 370 585
  * year         (year) int64 160B 2005 2010 2015 2020 ... 2085 2090 2095 2100
    spatial_ref  int32 4B 0
Attributes:
    author:            Zhaoquan.YU
    change_method:     change
    coverage:          land
    create_time:       20/06/2024
    data_description:  The Bureau of Meteorology high-resolution Regional Rea...
    data_method:       
    data_reference:    Global Change Mast

#### 1.2 Stats sub level for Tropical_Cyclone_Frequency Data

In [ ]:
# running (base) johnny@johnny-worker-01

vns = [
    'Tropical_Cyclone_Frequency',
]

for vn in vns:
    input_path = f'http://192.168.1.177:9000/climatehub.climate.indicator.v1/Global'
    output_path = f'/home/johnny/Downloads/Data'
    os.makedirs(output_path, exist_ok=True)
    da = xr.open_zarr(f'{input_path}/Global_{vn}.zarr').rename({'lon':'x', 'lat':'y'}).rio.write_crs(crs)[vn]
    df_list = []
    ofn = f'{output_path}/{vn}_stats_sub_level.csv'

    if 'cat' in da.dims:
        for ssp in ssps:
            for year in years:
                for pth in [50]:
                    for cat in da.cat.values.tolist():
                        print(vn, ssp, year, pth, cat)
                        df = edge_track_zonal_stats(da.sel(ssp=ssp, year=year, pth=pth, cat=cat).squeeze(), shp, keep_cols=keep_cols)
                        df.insert(len(keep_cols), 'Indicator', vn)
                        df.insert(len(keep_cols), 'cat', cat)
                        df.insert(len(keep_cols), 'pth', pth)
                        df.insert(len(keep_cols), 'year', year)
                        df.insert(len(keep_cols), 'ssp', ssp)
                        df_list.append(df)
    else:
        for ssp in ssps:
            for year in years:
                for pth in pths:
                    print(vn, ssp, year, pth)
                    df = edge_track_zonal_stats(da.sel(ssp=ssp, year=year, pth=pth).squeeze(), shp, keep_cols=keep_cols)
                    df.insert(len(keep_cols), 'Indicator', vn)
                    df.insert(len(keep_cols), 'pth', pth)
                    df.insert(len(keep_cols), 'year', year)
                    df.insert(len(keep_cols), 'ssp', ssp)
                    df_list.append(df)
    pd.concat(df_list).to_csv(ofn, index=False)


#### 1.3 Stats sub level for Extreme_Water_Level Data

In [ ]:
# running (base) johnny@johnny-worker-01

vns = [
    'Extreme_Water_Level', 
]

for vn in vns:
    input_path = f'http://192.168.1.177:9000/climatehub.climate.indicator.v1/Global'
    output_path = f'/home/johnny/Downloads/Data'
    os.makedirs(output_path, exist_ok=True)
    da = xr.open_zarr(f'{input_path}/Global_{vn}.zarr').rename({'lon':'x', 'lat':'y'}).rio.write_crs(crs)[vn]
    df_list = []
    ofn = f'{output_path}/{vn}_stats_sub_level.csv'

    if 'cat' in da.dims:
        for ssp in ssps:
            for year in years:
                for pth in pths:
                    for cat in da.cat.values.tolist():
                        print(vn, ssp, year, pth, cat)
                        df = edge_track_zonal_stats(da.sel(ssp=ssp, year=year, pth=pth, cat=cat).squeeze(), shp, keep_cols=keep_cols)
                        df.insert(len(keep_cols), 'Indicator', vn)
                        df.insert(len(keep_cols), 'cat', cat)
                        df.insert(len(keep_cols), 'pth', pth)
                        df.insert(len(keep_cols), 'year', year)
                        df.insert(len(keep_cols), 'ssp', ssp)
                        df_list.append(df)
    else:
        for ssp in ssps:
            for year in years:
                for pth in pths:
                    print(vn, ssp, year, pth)
                    df = edge_track_zonal_stats(da.sel(ssp=ssp, year=year, pth=pth).squeeze(), shp, keep_cols=keep_cols)
                    df.insert(len(keep_cols), 'Indicator', vn)
                    df.insert(len(keep_cols), 'pth', pth)
                    df.insert(len(keep_cols), 'year', year)
                    df.insert(len(keep_cols), 'ssp', ssp)
                    df_list.append(df)
    pd.concat(df_list).to_csv(ofn, index=False)


#### 1.4 Stats sub level for SPEI_Drought_Probability_3mon Score

In [ ]:
# running (base) johnny@johnny-worker-01

vns = [
    'SPEI_Drought_Probability_3mon',
]

for vn in vns:
    input_path = f'/media/Public2/APIs_Published_Data/API_v3_for_Projects/Aggregated_Risk_Score'
    output_path = f'/media/Public4/project_output/Ares_portfolio_risk_assessment/output/Stats_countries_level/Score'
    os.makedirs(output_path, exist_ok=True)
    da = xPreprocess(xr.open_mfdataset(f'{input_path}/Global_{vn}*.nc4')).rename({'lon':'x', 'lat':'y'}).rio.write_crs("epsg:4326").score
    df_list = []
    ofn = f'{output_path}/{vn}_stats_sub_level_score.csv'
    if 'cat' in da.dims:
        for ssp in ssps:
            for year in years:
                for pth in pths:
                    for cat in da.cat.values.tolist():
                        print(vn, ssp, year, pth, cat)
                        df = edge_track_zonal_stats(da.sel(ssp=ssp, year=year, pth=pth, cat=cat).squeeze(), shp, keep_cols=keep_cols)
                        df.insert(len(keep_cols), 'Indicator', vn)
                        df.insert(len(keep_cols), 'cat', cat)
                        df.insert(len(keep_cols), 'pth', pth)
                        df.insert(len(keep_cols), 'year', year)
                        df.insert(len(keep_cols), 'ssp', ssp)
                        df_list.append(df)
    else:
        for ssp in ssps:
            for year in years:
                for pth in pths:
                    print(vn, ssp, year, pth)
                    df = edge_track_zonal_stats(da.sel(ssp=ssp, year=year, pth=pth).squeeze(), shp, keep_cols=keep_cols)
                    df.insert(len(keep_cols), 'Indicator', vn)
                    df.insert(len(keep_cols), 'pth', pth)
                    df.insert(len(keep_cols), 'year', year)
                    df.insert(len(keep_cols), 'ssp', ssp)
                    df_list.append(df)
    pd.concat(df_list).to_csv(ofn, index=False)


#### For Extreme_Precipitation & Extreme_Wind_Speed

In [12]:
import numpy as np, xarray as xr, os, pandas as pd, geopandas as gpd
import os, requests; exec(requests.get(f'http://45.120.118.102:6888/lib/Jpkg?key=J.').json()['Jpkg'])

crs = "epsg:4326"
shp = gpd.read_file(r'M:\Project_Output_2025_10\MAM-Demo\Dummy_Polygons\info\info_shp\info_shp.shp').to_crs(crs) 
shp = shp.sort_values('source') 
shp['SN'] = range(1, len(shp)+1)

keep_cols = ['SN', 'POA_CODE21', 'source']
for col in keep_cols:
    shp[col] = shp[col].astype(str)

print('Invalid records:', shp.loc[~shp.geometry.is_valid])

# ==================== 1. SETTING 配置区域 ====================
aris  = [100]                   # ari 维度 (年一遇重现期)
# hrs   = 24                           # 固定保留 hrs = 24 小时历时
pths  = [5, 50, 95]
ssps  = [119, 126, 245, 370, 585]
years = [2005, 2030, 2040, 2050]

vns = [
    "Extreme_Wind_Speed", 
]
# ============================================================

for vn in vns:
    input_path = f"http://192.168.1.218:9000/climatehub.climate.indicator.v1/Australia" 
    output_path = r"Z:\works\Lucas\Data_Works" 
    os.makedirs(output_path, exist_ok=True)
    
    da = (
        xr.open_zarr(f"{input_path}/Australia_{vn}.zarr") 
        .rename({"lon": "x", "lat": "y"})
        .rio.write_crs(crs)[vn]
    )

    print("Zarr 数据的结构和维度信息如下：")
    print(da)

    df_list = []
    ofn = f"{output_path}/{vn}_stats_sub_level.csv"

    # 根据数据是否包含 cat 维度，走不同的循环逻辑
    if "cat" in da.dims:
        for ssp in ssps:
            for year in years:
                for pth in pths:
                    for ari in aris:       
                        for cat in da.cat.values.tolist():
                            print(f"Processing: {vn} | ssp:{ssp} | year:{year} | pth:{pth} | ari:{ari} | cat:{cat}")
                            
                            # 1. 精准选择目标维度（显式包含 hrs=hrs）
                            temp_da = da.sel(ssp=ssp, year=year, pth=pth, ari=ari, cat=cat)
                            
                            # 2. 安全防御：清除其他多余维度（如有）
                            extra_dims = [d for d in temp_da.dims if d not in ['x', 'y']]
                            if extra_dims:
                                temp_da = temp_da.isel({d: 0} for d in extra_dims)
                            data_input = temp_da.squeeze()

                            df = edge_track_zonal_stats(
                                data_input,
                                shp,
                                keep_cols=keep_cols,
                            )
                            # 3. 结果中插入相应的属性列
                            df.insert(len(keep_cols), "Indicator", vn)
                            df.insert(len(keep_cols), "cat", cat)
                            # df.insert(len(keep_cols), "hrs", hrs) # 新增 hrs 列到 CSV 结果
                            df.insert(len(keep_cols), "ari", ari) 
                            df.insert(len(keep_cols), "pth", pth)
                            df.insert(len(keep_cols), "year", year)
                            df.insert(len(keep_cols), "ssp", ssp)
                            df_list.append(df)
    else:
        for ssp in ssps:
            for year in years:
                for pth in pths:
                    for ari in aris:       
                        print(f"Processing: {vn} | ssp:{ssp} | year:{year} | pth:{pth} | ari:{ari}")
                        
                        # 1. 精准选择目标维度（显式包含 hrs=hrs）
                        temp_da = da.sel(ssp=ssp, year=year, pth=pth, ari=ari)
                        
                        # 2. 安全防御：清除其他多余维度
                        extra_dims = [d for d in temp_da.dims if d not in ['x', 'y']]
                        for d in extra_dims:
                            temp_da = temp_da.isel({d: 0})
                        data_input = temp_da.squeeze()

                        df = edge_track_zonal_stats(
                            data_input,
                            shp,
                            keep_cols=keep_cols,
                        )
                        # 3. 结果中插入相应的属性列
                        df.insert(len(keep_cols), "Indicator", vn)
                        # df.insert(len(keep_cols), "hrs", hrs)     # 新增 hrs 列到 CSV 结果
                        df.insert(len(keep_cols), "ari", ari)     
                        df.insert(len(keep_cols), "pth", pth)
                        df.insert(len(keep_cols), "year", year)
                        df.insert(len(keep_cols), "ssp", ssp)
                        df_list.append(df)

    # 保存最终表格
    if df_list:
        pd.concat(df_list).to_csv(ofn, index=False)
        print(f"成功保存统计结果至: {ofn}")

Invalid records: Empty GeoDataFrame
Columns: [POA_CODE21, POA_NAME21, AUS_CODE21, AUS_NAME21, AREASQKM21, LOCI_URI21, SHAPE_Leng, SHAPE_Area, SN, source, geometry]
Index: []
Zarr 数据的结构和维度信息如下：
<xarray.DataArray 'Extreme_Wind_Speed' (ari: 1, pth: 3, ssp: 5, year: 20,
                                        y: 691, x: 886)> Size: 735MB
dask.array<open_dataset-Extreme_Wind_Speed, shape=(1, 3, 5, 20, 691, 886), dtype=float32, chunksize=(1, 3, 5, 20, 120, 30), chunktype=numpy.ndarray>
Coordinates:
  * ari          (ari) int64 8B 100
  * y            (y) float32 3kB -10.0 -10.05 -10.1 ... -44.4 -44.45 -44.5
  * x            (x) float32 4kB 112.0 112.0 112.1 112.1 ... 156.1 156.2 156.2
  * pth          (pth) int64 24B 5 50 95
  * ssp          (ssp) int64 40B 119 126 245 370 585
  * year         (year) int64 160B 2005 2010 2015 2020 ... 2085 2090 2095 2100
    spatial_ref  int32 4B 0
Attributes:
    author:            Zhaoquan.YU
    change_method:     change
    coverage:          land
    cr